# F-OAL — Lab 2: Empirical Demonstration & Critical Stress-Testing

**Học phần:** Introduction to Machine Learning (HCMUS)  
**Mục tiêu:** Chạy trọn bộ thực nghiệm **E1 (Long-Tail OCIL)**, **E2 (Sensor Corruptions)**, và **E3 ($\mathcal{O}(D^2)$ Scaling & $(D \times \gamma)$ Over-Parameterization)** trên GPU (Colab / Kaggle Tesla T4) cho cả **CIFAR-100**, **DTD**, và **FGVC-Aircraft**, đồng bộ trực tiếp lên project W&B công khai `hcmus-foal-reproduction` với bộ tags chuẩn `['lab2-empirical-demo', 'critical-analysis']`.

In [ ]:
# 1. Clone / cập nhật repository và cài đặt dependencies
import os, subprocess
if not os.path.exists('F-OAL'):
    subprocess.run(['git', 'clone', 'https://github.com/caoban123/F-OAL.git'], check=True)
%cd F-OAL
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

In [ ]:
# 2. Đăng nhập Weights & Biases (tất cả run của Lab 2 tự động gắn tag 'lab2-empirical-demo')
import wandb, torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
wandb.login()

In [ ]:
# 3. Chạy E1 (Long-Tail OCIL — 3 seeds: 42, 123, 2026) trên cả 3 bộ dữ liệu
for ds in ['dtd', 'aircraft', 'cifar100']:
    !python -m scripts.run_lab2_longtail --dataset {ds} --seeds 42 123 2026 --batch-size 25 --cache-dir ./lab2_feature_cache --wandb

In [ ]:
# 4. Chạy E3 (Scaling D & Lưới (D x gamma) + FD2 Adaptive Ridge) trên cả 3 bộ dữ liệu
for ds in ['dtd', 'aircraft', 'cifar100']:
    !python -m scripts.run_lab2_scaling_sensitivity --dataset {ds} --seeds 42 123 2026 --cache-dir ./lab2_feature_cache --wandb

In [ ]:
# 5. Chạy E2 (Covariate-Shift / Sensor Corruptions s = 0..5) trên DTD, Aircraft, CIFAR-100
!python -m scripts.run_lab2_corruptions --dataset dtd --device cuda --wandb
!python -m scripts.run_lab2_corruptions --dataset aircraft --device cuda --wandb
!python -m scripts.run_lab2_corruptions --dataset cifar100 --severities 0 1 3 5 --limit-train-per-class 100 --device cuda --wandb

In [ ]:
# 6. Xuất toàn bộ biểu đồ chuẩn 300 DPI cho Poster khổ dọc A2 và nén kết quả tải về
!python -m scripts.generate_lab2_figures --input-root results/lab2_empirical --output-dir results/lab2_empirical/figures
!zip -r foal_lab2_empirical_results.zip results/lab2_empirical/